# Notebook de documentacion del datasey y tratamiento datos.


## Equipo

- Alumno 1 : Norberto Narvaez
- Alumno 2 :


In [1]:
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

splits = pd.read_csv("data/splits.csv")
print(splits["split"].value_counts())
print("PyTorch:", torch.__version__, "| ¿GPU?:", torch.cuda.is_available())

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


split
train          9581
test_open      3105
test_closed    1986
val            1978
Name: count, dtype: int64
PyTorch: 2.11.0+cu130 | ¿GPU?: False


el Dataset y los DataLoaders


In [2]:
personas_train = sorted(splits.loc[splits["split"] == "train", "identidad"].unique())
clase_de = {nombre: i for i, nombre in enumerate(personas_train)}

aug_train = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-10, 10), translate_percent=(-0.05, 0.05), scale=(0.95, 1.05), p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.GaussNoise(std_range=(0.02, 0.06), p=0.2),
    A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.1, 0.25), hole_width_range=(0.1, 0.25), p=0.3),
    A.ImageCompression(quality_range=(50, 95), p=0.3),
], seed=SEED)


class CarasDataset(Dataset):
    def __init__(self, tabla, augmentacion=None):
        self.rutas = tabla["path_alineada"].tolist()
        self.etiquetas = [clase_de.get(n, -1) for n in tabla["identidad"]]
        self.augmentacion = augmentacion

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, i):
        img = cv2.cvtColor(cv2.imread(self.rutas[i]), cv2.COLOR_BGR2RGB)
        if self.augmentacion is not None:
            img = self.augmentacion(image=img)["image"]
        img = (img.astype(np.float32) - 127.5) / 127.5
        img = torch.from_numpy(img).permute(2, 0, 1)
        return img, self.etiquetas[i]


train_ds = CarasDataset(splits[splits["split"] == "train"], aug_train)
val_ds = CarasDataset(splits[splits["split"] == "val"])

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
val_dl = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=0)

x, y = next(iter(train_dl))
print("Lote de imágenes:", x.shape)
print("Etiquetas:", y[:10])
print("Clases:", len(personas_train))

Lote de imágenes: torch.Size([64, 3, 112, 112])
Etiquetas: tensor([31, 62, 51, 48, 73, 78,  0, 20, 76, 83])
Clases: 85


In [3]:
def bloque(entrada, salida):
    return nn.Sequential(
        nn.Conv2d(entrada, salida, kernel_size=3, padding=1),
        nn.BatchNorm2d(salida),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class CNNCaras(nn.Module):
    def __init__(self, n_clases, dim_emb=128):
        super().__init__()
        self.features = nn.Sequential(
            bloque(3, 32),
            bloque(32, 64),
            bloque(64, 128),
            bloque(128, 256),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.embedding = nn.Linear(256, dim_emb)
        self.dropout = nn.Dropout(0.3)
        self.clasificador = nn.Linear(dim_emb, n_clases)

    def obtener_embedding(self, x):
        x = self.features(x)
        x = self.pool(x).flatten(1)
        return self.embedding(x)

    def forward(self, x):
        emb = self.obtener_embedding(x)
        return self.clasificador(self.dropout(emb))


modelo = CNNCaras(n_clases=len(personas_train))
print(modelo)
print("Parámetros:", sum(p.numel() for p in modelo.parameters()))
print("Salida para un lote:", modelo(x).shape)

CNNCaras(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): Conv2d(128, 256, kernel_size=(3, 3

In [ ]:
import time
import copy

EPOCAS = 30
PACIENCIA = 5
LR = 1e-3
MODELO_FINAL = Path("models/cnn_caras_final.pth")

criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode="min", factor=0.5, patience=2)


def correr_epoca(dl, entrenar):
    modelo.train(entrenar)
    total_loss, aciertos, total = 0.0, 0, 0
    with torch.set_grad_enabled(entrenar):
        for x, y in dl:
            salida = modelo(x)
            loss = criterio(salida, y)
            if entrenar:
                optimizador.zero_grad()
                loss.backward()
                optimizador.step()
            total_loss += loss.item() * len(y)
            aciertos += (salida.argmax(1) == y).sum().item()
            total += len(y)
    return total_loss / total, aciertos / total


if MODELO_FINAL.exists():
    modelo.load_state_dict(torch.load(MODELO_FINAL))
    historial = pd.read_csv("data/historial_cnn.csv").to_dict("list")
    print("Modelo cargado desde", MODELO_FINAL)
    print(pd.DataFrame(historial).round(3))
else:
    historial = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    mejor_val_loss = float("inf")
    epocas_sin_mejora = 0

    for epoca in range(1, EPOCAS + 1):
        inicio = time.time()
        tr_loss, tr_acc = correr_epoca(train_dl, entrenar=True)
        va_loss, va_acc = correr_epoca(val_dl, entrenar=False)
        scheduler.step(va_loss)

        historial["train_loss"].append(tr_loss)
        historial["val_loss"].append(va_loss)
        historial["train_acc"].append(tr_acc)
        historial["val_acc"].append(va_acc)
        pd.DataFrame(historial).to_csv("data/historial_cnn.csv", index=False)

        print(f"Época {epoca:2d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
              f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time() - inicio:.0f} s")

        if va_loss < mejor_val_loss:
            mejor_val_loss = va_loss
            mejor_estado = copy.deepcopy(modelo.state_dict())
            torch.save(mejor_estado, "models/cnn_caras_mejor.pth")
            epocas_sin_mejora = 0
        else:
            epocas_sin_mejora += 1
            if epocas_sin_mejora >= PACIENCIA:
                print("Early stopping: la validación no mejoró en", PACIENCIA, "épocas")
                break

    modelo.load_state_dict(mejor_estado)
    torch.save(modelo.state_dict(), MODELO_FINAL)

KeyboardInterrupt: 